## gpcp, era 데이터 처리 코드

In [ ]:
def save_frames_as_images(ds, save_folder, img_size, meta_path):
    # 동아시아 영역 자르기 (위도 방향 주의)
    ds_eastasia = ds.sel(lat=slice(33, 43), lon=slice(124, 132))

    # lev 차원 제거 (있으면)
    if 'lev' in ds_eastasia['precipitation'].dims:
        data = ds_eastasia['precipitation'].squeeze('lev').values
    else:
        data = ds_eastasia['precipitation'].values
    
    # 결측치 -> 0
    data = np.nan_to_num(data, nan=0.0)

    # 최소/최대값
    min_val = float(data.min())
    max_val = float(data.max())

    # 0~255 정규화
    normed_data = (data - min_val) / (max_val - min_val) * 255
    normed_data = normed_data.astype(np.uint8)

    # 폴더 생성
    os.makedirs(save_folder, exist_ok=True)

    # 프레임 이미지 저장
    for idx, frame in enumerate(tqdm(normed_data)):
        img = Image.fromarray(np.flipud(frame))  # 위아래 반전
        img = img.resize(img_size, resample=Image.BILINEAR)
        img = img.convert('RGB')
        img.save(os.path.join(save_folder, f"{idx:05d}.png"))

    # 메타데이터 저장
    metadata = {
        "min_val": min_val,
        "max_val": max_val,
        'uint': "mm/day",
        "lat_range": [float(ds_eastasia.lat.max()), float(ds_eastasia.lat.min())],
        "lon_range": [float(ds_eastasia.lon.min()), float(ds_eastasia.lon.max())],
        'original_shape': list(data.shape),
        'time': [str(t) for t in ds_eastasia.time.values]
    }

    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=4)


## IMERG 이미지 처리 코드
### IMERG는 dim이 (time, lon, lat) 이어서 transpose 후 (time, lat, lon)으로 설정

In [ ]:
# imerg 
import os, json
import numpy as np
from tqdm import tqdm
from PIL import Image

def save_frames_as_images(ds, save_folder, img_size, meta_path):
    # 사용할 변수 선택
    da = ds['precipitation']      # DataArray

    # 동아시아 영역 자르기
    da = da.sel(lat=slice(33, 43), lon=slice(124, 134))

    # lev 차원 제거 (있으면)
    if 'lev' in da.dims:
        da = da.squeeze('lev')

    # 차원 순서를 (time, lat, lon)으로 통일
    # IMERG는 보통 (time, lon, lat)이니까 여기서 순서 맞춰줌
    want_dims = [d for d in ['time', 'lat', 'lon'] if d in da.dims]
    da = da.transpose(*want_dims)

    # 위도를 북→남(큰 위도→작은 위도) 순서로 정렬
    # 그림 그릴 때 위쪽이 북쪽이 되게 하고 싶으면 켜기
    if da['lat'][0] < da['lat'][-1]:
        da = da.sortby('lat', ascending=False)

    # numpy 배열로 변환: (time, lat, lon)
    data = da.values

    # 결측치 → 0
    data = np.nan_to_num(data, nan=0.0)

    # 최소/최대값
    min_val = float(data.min())
    max_val = float(data.max())

    # 0~255로 정규화
    normed_data = (data - min_val) / (max_val - min_val + 1e-12) * 255
    normed_data = normed_data.astype(np.uint8)

    # 폴더 생성
    os.makedirs(save_folder, exist_ok=True)
    os.makedirs(os.path.dirname(meta_path), exist_ok=True)

    # 프레임 이미지 저장
    for idx, frame in enumerate(tqdm(normed_data)):
        # 이제 frame.shape == (lat, lon) == (H, W)  ✅
        img = Image.fromarray(frame)
        img = img.resize(img_size, resample=Image.BILINEAR)
        img = img.convert('L')
        img.save(os.path.join(save_folder, f"{idx:05d}.png"))

    # 메타데이터 저장
    metadata = {
        "min_val": min_val,
        "max_val": max_val,
        "unit": "mm/day",
        "lat_range": [float(da['lat'].max()), float(da['lat'].min())],
        "lon_range": [float(da['lon'].min()), float(da['lon'].max())],
        "original_shape": list(data.shape),  # (time, lat, lon)
        "time": [str(t) for t in da['time'].values]
    }

    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=4)


In [ ]:
# ds_lr = xr.open_dataset(LR_PATH)
# ds_hr = xr.open_dataset(HR_PATH)
import os
import json
import numpy as np
import xarray as xr
from tqdm import tqdm
from PIL import Image

path = "../../../data/IMERG/Imerg_raw/imerg_merged_precip.nc"
ds = xr.open_dataset(path)

SAVE_DIR = "./load/climate"

# === Train ===
train_time_slice = slice('1998-01-01', '2009-12-31')
save_frames_as_images(
    ds.sel(time=train_time_slice),
    save_folder=os.path.join(SAVE_DIR, 'climate_train_korea_imerg50_test'),
    img_size=(50, 50),
    meta_path=os.path.join(SAVE_DIR, 'meta', 'train_imerg50_meta_test.json')
)

save_frames_as_images(
    ds.sel(time=train_time_slice),
    save_folder=os.path.join(SAVE_DIR, 'climate_train_korea_imerg100_test'),
    img_size=(100, 100),
    meta_path=os.path.join(SAVE_DIR, 'meta', 'train_imerg100_meta_test.json')
)


# # === Validation ===
val_time_slice = slice('2010-01-01', '2019-12-31')
save_frames_as_images(
    ds.sel(time=val_time_slice),
    save_folder=os.path.join(SAVE_DIR, 'climate_val_korea_imerg50_test'),
    img_size=(50, 50),
    meta_path=os.path.join(SAVE_DIR, 'meta', 'val_imerg50_meta_test.json')
)
save_frames_as_images(
    ds.sel(time=val_time_slice),
    save_folder=os.path.join(SAVE_DIR, 'climate_val_korea_imerg100_test'),
    img_size=(100, 100),
    meta_path=os.path.join(SAVE_DIR, 'meta', 'val_imerg100_meta_test.json')
)




100%|██████████| 3652/3652 [00:01<00:00, 2413.11it/s]
